### Imports

In [1]:
from rdfine import GraphReader, infer
from compilers import PipelineExtractor, PipelineAssembler, LdioConfigCompiler, RdfcConfigCompiler, DockerComposeCompiler
from rdflib import Graph 

### Load the graph

In [2]:
# Loading the graph
input_folder = "..\\data\\"
catalog_graph = Graph()
catalog_graph.parse(input_folder + "graph.ttl", publicID="file:///workspace/pipeline/")
graph_reader = GraphReader(catalog_graph)
print(f"{len(graph_reader.get_triples())} triples loaded.")
new_graph = graph_reader.to_graph() # This is important, inference cannot handle blank nodes

313 triples loaded.


### Enrich the graph via inference rules

In [3]:
# Applying inference rules
enriched_graph = infer(new_graph, input_folder + "inference_rules.yaml", max_repetitions=10)
graph_reader = GraphReader(enriched_graph)
print(f"{len(graph_reader.get_triples())} triples in total after applying inference rules.")

359 triples in total after applying inference rules.


### Extract a single pipeline 

In [4]:
extracted_pipeline = PipelineExtractor(graph=enriched_graph, pipeline_id=":InteroperablePipeline").compile()

### Assemble the pipeline

In [5]:
assembled_pipeline = PipelineAssembler(extracted_pipeline).compile()

### Generate the LDIO config

In [6]:
ldio_config = LdioConfigCompiler(assembled_pipeline).compile()
print(ldio_config)

name: LdioHttpInPipeline
input:
  name: Ldio:HttpIn
outputs:
- name: Ldio:ConsoleOut
  config:
    rdf-writer:
      content-type: text/turtle



### Generate the RDF Connect config

In [7]:
rdfc_config = RdfcConfigCompiler(assembled_pipeline).compile()
print(rdfc_config)

@prefix : <http://example.org/example/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfc: <https://w3id.org/rdf-connect#> .

<> a rdfc:Pipeline ;
    owl:imports <file:///usr/local/lib/python3.13/site-packages/rdfc_http_out/processor.ttl>,
        <file:///usr/local/lib/python3.13/site-packages/rdfc_runner/index.ttl>,
        <file:///workspace/pipeline/node_modules/@rdfc/http-utils-processor-ts/processors.ttl>,
        <file:///workspace/pipeline/node_modules/@rdfc/js-runner/index.ttl> ;
    rdfc:consistsOf :env_1,
        :env_2 .

:RdfcHttpOutStep a rdfc:HttpOut ;
    rdfc:endpoint "http://ldio-workbench:8080/LdioHttpInPipeline" ;
    rdfc:reader :channel_1 .

:RdfcPollApiStep a rdfc:HttpFetch ;
    rdfc:cron "*/10 * * * * *" ;
    rdfc:url "https://dishacled-api.azurewebsites.net/api/v1/source-a/current" ;
    rdfc:writer :channel_1 .

:env_1 rdfc:instantiates rdfc:NodeRunner ;
    rdfc:processor :RdfcPollApiStep .

:env_2 rdfc:instantiates rdfc:PyRunner ;
    rdfc:pr

### Generate the DockerCompose Config

In [10]:
docker_compose_config = DockerComposeCompiler(assembled_pipeline).compile()
print(docker_compose_config)

rdf-connect:
  container_name: rdf-connect
  image: rdf-connect:latest
  build: ../../resources/rdfc-docker
  volumes:
  - ./rdfc_pipeline.ttl:/workspace/pipeline/pipeline.ttl:ro
  environment:
    LOG_LEVEL: debug
  command: npx rdfc /workspace/pipeline/pipeline.ttl
ldio-pipeline-starter:
  image: curlimages/curl
  volumes:
  - ./ldio_pipeline.yml:/pipeline.yml:ro
  command: 'sh -c " sleep 30 && curl -X POST -H ''content-type: application/yaml''
    http://ldio-workbench:8080/admin/api/v1/pipeline --data-binary @/pipeline.yml
    "'
ldio-workbench:
  container_name: ldio-workbench
  image: ldes/ldi-orchestrator:2.8.0-SNAPSHOT
  ports:
  - 8080:8080

